# Data Analysis with Pandas, Matplotlib & Seaborn
### A hands-on tutorial using a real vegetation survey dataset

**Dataset:** `draft_list.xlsx` — a plant survey recorded along 5 transects, with
plant counts taken **pre-monsoon** and **post-monsoon**.

Columns:
| Column | Meaning |
|---|---|
| `Transect` | ID of the survey transect / plot (1–5) |
| `PLANT` | Species name of the plant recorded |
| `pre monsoon` | Number of individuals counted before the monsoon |
| `post monsoon` | Number of individuals counted after the monsoon |

This notebook walks through a complete, beginner-friendly data analysis workflow:

1. **Setup** – importing libraries
2. **Loading & inspecting data** with pandas
3. **Cleaning** the data
4. **Exploring** the data with pandas (filtering, grouping, pivoting)
5. **Visualizing** with matplotlib (the fundamentals)
6. **Visualizing** with seaborn (statistical plots)
7. **Putting it together** — a mini end-to-end analysis

Every code cell has inline comments explaining *what* each line does and *why*,
so this notebook can be used as a self-contained learning resource.


## 1. Setup — importing the libraries

We need three libraries for this tutorial:

- **pandas** → loading, cleaning and manipulating tabular data
- **matplotlib** → the foundational plotting library in Python
- **seaborn** → a statistical plotting library built on top of matplotlib (nicer defaults + easier syntax for common statistical charts)

In [ ]:
# pandas is the main library for working with tabular (spreadsheet-like) data
import pandas as pd

# numpy is a numerical computing library; pandas is built on top of it
# we'll use it occasionally for numeric helpers (e.g. np.round)
import numpy as np

# matplotlib.pyplot is the standard interface for creating plots
import matplotlib.pyplot as plt

# seaborn is built on matplotlib and gives us attractive, statistics-aware charts
import seaborn as sns

# This "magic command" tells Jupyter to display plots directly inside the notebook
# (not needed in newer Jupyter versions, but included for compatibility)
%matplotlib inline

# Set a global seaborn theme so every plot in this notebook looks consistent
sns.set_theme(style="whitegrid")

# Set a default figure size so we don't have to repeat it in every plot
plt.rcParams["figure.figsize"] = (8, 5)

print("Libraries imported successfully!")
print("pandas version:", pd.__version__)


## 2. Loading the data

We read the Excel file into a pandas **DataFrame** — the core pandas object, essentially a table with labeled rows and columns.

In [ ]:
# pd.read_excel() reads an Excel (.xlsx) file into a DataFrame
# make sure "draft_list.xlsx" is in the same folder as this notebook
df = pd.read_excel("draft_list.xlsx")

# .head() shows the first 5 rows by default — a quick sanity check
df.head()


In [ ]:
# .shape returns a tuple: (number_of_rows, number_of_columns)
print("Rows, Columns:", df.shape)

# .columns lists all column names — useful to spot typos / stray spaces
print("Columns:", list(df.columns))


Notice that the first column name has a trailing space: `'Transect '`. 
This is extremely common in real-world spreadsheets and is exactly the kind of thing we fix during **data cleaning**.

## 3. Cleaning the data

Real datasets are rarely perfectly tidy. Before analyzing, we always check for:
- messy column names (extra spaces, inconsistent case)
- missing values
- duplicate rows
- incorrect data types

In [ ]:
# .str.strip() removes leading/trailing whitespace from each column name
# list comprehension applies this to every column name in df.columns
df.columns = [c.strip() for c in df.columns]

# Confirm the fix worked
print("Cleaned columns:", list(df.columns))


In [ ]:
# .info() gives a compact summary: column names, non-null counts, and dtypes
# Great first check for missing values and wrong data types
df.info()


In [ ]:
# .isnull() flags each cell as True/False (missing or not)
# .sum() then adds up True values per column -> total missing values per column
df.isnull().sum()


In [ ]:
# .duplicated() flags rows that are exact repeats of an earlier row
# .sum() counts how many such duplicate rows exist
duplicate_count = df.duplicated().sum()
print(f"Number of fully duplicated rows: {duplicate_count}")

# Good news: this dataset has no missing values and no duplicate rows,
# so we can move straight to exploration.


In [ ]:
# .describe() gives quick summary statistics (count, mean, std, min, quartiles, max)
# for all NUMERIC columns — a fast way to understand the scale/spread of the data
df.describe()


## 4. Exploring the data with pandas

Now that the data is clean, let's answer some real questions using core pandas operations: **selecting, filtering, sorting, grouping, and pivoting**.

### 4.1 Selecting & filtering rows

In [ ]:
# Select a single column -> returns a pandas Series
species_column = df["PLANT"]
species_column.head()


In [ ]:
# Boolean filtering: keep only rows where Transect == 1
# df["Transect"] == 1 creates a True/False mask; df[mask] applies it
transect_1 = df[df["Transect"] == 1]
print(f"Transect 1 has {len(transect_1)} recorded plant entries")
transect_1.head()


In [ ]:
# Combine conditions with & (and) / | (or). Each condition must be in parentheses.
# Here: entries where MORE plants were counted post-monsoon than pre-monsoon
increased = df[df["post monsoon"] > df["pre monsoon"]]
print(f"{len(increased)} species entries increased after the monsoon")
increased.head()


### 4.2 Creating new (derived) columns

In [ ]:
# We can create a new column from existing ones using simple arithmetic.
# This computes the CHANGE in count from pre- to post-monsoon.
df["change"] = df["post monsoon"] - df["pre monsoon"]

# A second derived column: percentage change (guard against divide-by-zero with np.where)
df["pct_change"] = np.where(
    df["pre monsoon"] == 0,      # condition: was the pre-monsoon count zero?
    np.nan,                      # if yes -> percentage change is undefined (NaN)
    (df["change"] / df["pre monsoon"]) * 100   # otherwise -> normal % change formula
)

df.head()


### 4.3 Sorting

In [ ]:
# .sort_values() orders rows by one or more columns.
# ascending=False means largest values first.
top_increases = df.sort_values("change", ascending=False).head(10)
top_increases[["Transect", "PLANT", "pre monsoon", "post monsoon", "change"]]


### 4.4 Grouping — the pandas `groupby` workflow

`groupby` follows a **split → apply → combine** pattern:
1. **Split** the data into groups (e.g. by Transect)
2. **Apply** a function to each group (e.g. sum, mean)
3. **Combine** the results back into a single table

In [ ]:
# Group all rows by Transect, then sum the numeric columns within each group
# This tells us total plant counts recorded per transect
transect_totals = df.groupby("Transect")[["pre monsoon", "post monsoon"]].sum()
transect_totals


In [ ]:
# We can also group by species to see which plants were recorded most often
# .size() counts the number of rows (entries) per group
species_frequency = df.groupby("PLANT").size().sort_values(ascending=False)

# Show the 10 most frequently recorded species across all transects
species_frequency.head(10)


In [ ]:
# .agg() lets us apply MULTIPLE summary functions at once, per group
transect_summary = df.groupby("Transect")["change"].agg(["mean", "sum", "min", "max"])

# .round(2) keeps the output tidy to 2 decimal places
transect_summary.round(2)


### 4.5 Pivot tables

A pivot table reshapes data: it takes unique values from one column and spreads them into new columns, aggregating another value inside the grid.

In [ ]:
# pivot_table: rows = PLANT, columns = Transect, values = pre-monsoon count
# aggfunc="sum" combines counts if a species+transect combination has multiple rows
pivot = df.pivot_table(
    index="PLANT",
    columns="Transect",
    values="pre monsoon",
    aggfunc="sum",
    fill_value=0          # replace missing combinations with 0 instead of NaN
)

pivot.head(10)


## 5. Visualizing with Matplotlib

Matplotlib is the foundation: nearly every other Python plotting library (including seaborn) is built on top of it. Understanding its basics — **Figure** and **Axes** — makes every other library easier to use.

- **Figure** = the whole canvas/window
- **Axes** = an individual plot inside that canvas (a figure can hold several axes/subplots)

### 5.1 A basic bar chart

In [ ]:
# plt.subplots() creates a Figure and one Axes object in a single call
# figsize controls the width and height of the figure in inches
fig, ax = plt.subplots(figsize=(8, 5))

# .plot(kind="bar") on our grouped Series draws a bar chart on the given Axes
transect_totals["pre monsoon"].plot(kind="bar", ax=ax, color="seagreen")

# Always label your axes and give the plot a title — good practice for any chart
ax.set_title("Total Pre-Monsoon Plant Counts by Transect")
ax.set_xlabel("Transect")
ax.set_ylabel("Total count")

# tight_layout prevents labels from being cut off
plt.tight_layout()
plt.show()


### 5.2 Comparing two series with a grouped bar chart

In [ ]:
# transect_totals already has both 'pre monsoon' and 'post monsoon' columns,
# so calling .plot(kind='bar') directly on the DataFrame draws grouped bars automatically
fig, ax = plt.subplots(figsize=(8, 5))

transect_totals.plot(kind="bar", ax=ax)

ax.set_title("Pre- vs Post-Monsoon Plant Counts by Transect")
ax.set_xlabel("Transect")
ax.set_ylabel("Total count")
ax.legend(title="Season")   # legend distinguishes the two bars

plt.tight_layout()
plt.show()


### 5.3 Histogram — understanding the distribution of a numeric column

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# A histogram bins numeric values and shows how many observations fall in each bin
# bins=10 splits the range of values into 10 equal-width buckets
ax.hist(df["pre monsoon"], bins=10, color="steelblue", edgecolor="black")

ax.set_title("Distribution of Pre-Monsoon Plant Counts")
ax.set_xlabel("Pre-monsoon count")
ax.set_ylabel("Frequency (number of species entries)")

plt.tight_layout()
plt.show()


### 5.4 Scatter plot — relationship between two numeric variables

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# A scatter plot places one dot per row: x = pre-monsoon count, y = post-monsoon count
ax.scatter(df["pre monsoon"], df["post monsoon"], alpha=0.6, color="darkorange")

# A reference diagonal line (y = x) helps visually separate increases from decreases:
# points ABOVE the line increased after the monsoon, points BELOW decreased
max_val = max(df["pre monsoon"].max(), df["post monsoon"].max())
ax.plot([0, max_val], [0, max_val], linestyle="--", color="gray", label="No change (y = x)")

ax.set_title("Pre-Monsoon vs Post-Monsoon Counts")
ax.set_xlabel("Pre-monsoon count")
ax.set_ylabel("Post-monsoon count")
ax.legend()

plt.tight_layout()
plt.show()


### 5.5 Subplots — multiple charts in one figure

In [ ]:
# plt.subplots(rows, cols) creates a grid of Axes objects
# Here we make a 1-row, 2-column grid -> axes is an array of 2 Axes objects
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left subplot: histogram of pre-monsoon counts
axes[0].hist(df["pre monsoon"], bins=10, color="seagreen", edgecolor="black")
axes[0].set_title("Pre-Monsoon Distribution")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Frequency")

# Right subplot: histogram of post-monsoon counts
axes[1].hist(df["post monsoon"], bins=10, color="tomato", edgecolor="black")
axes[1].set_title("Post-Monsoon Distribution")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 6. Visualizing with Seaborn

Seaborn works directly with pandas DataFrames and understands statistical concepts (categories, distributions, regressions) out of the box, usually needing far less code than matplotlib for the same statistical chart.

### 6.1 Boxplot — comparing distributions across categories

A boxplot summarizes a distribution using 5 numbers: minimum, 25th percentile, median, 75th percentile, and maximum (plus outliers as individual points).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# x = categorical column (Transect), y = numeric column (pre monsoon count)
# seaborn automatically groups the data by the x variable
sns.boxplot(data=df, x="Transect", y="pre monsoon", hue="Transect", legend=False, ax=ax, palette="Set2")

ax.set_title("Spread of Pre-Monsoon Counts across Transects")
ax.set_xlabel("Transect")
ax.set_ylabel("Pre-monsoon count")

plt.tight_layout()
plt.show()


### 6.2 Violin plot — boxplot + distribution shape combined

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# A violin plot is like a boxplot but also shows the full shape of the distribution
# (wider sections = more data points at that value)
sns.violinplot(data=df, x="Transect", y="change", hue="Transect", legend=False, ax=ax, palette="coolwarm")

# A horizontal reference line at 0 helps show which transects mostly increased vs decreased
ax.axhline(0, color="black", linestyle="--", linewidth=1)

ax.set_title("Distribution of Change (Post − Pre) by Transect")
ax.set_xlabel("Transect")
ax.set_ylabel("Change in count")

plt.tight_layout()
plt.show()


### 6.3 Bar plot with confidence intervals

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# sns.barplot automatically computes the MEAN per category and draws a confidence
# interval (the vertical black line) — matplotlib would need manual calculation for this
sns.barplot(data=df, x="Transect", y="pre monsoon", ax=ax, color="cornflowerblue", errorbar="sd")

ax.set_title("Mean Pre-Monsoon Count per Transect (± std dev)")
ax.set_xlabel("Transect")
ax.set_ylabel("Mean count")

plt.tight_layout()
plt.show()


### 6.4 Heatmap — visualizing the pivot table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

# sns.heatmap renders a 2D table as a color-coded grid — perfect for our pivot table
# annot=True writes the actual numbers inside each cell
# cmap sets the color palette; fmt="d" formats numbers as integers
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlGnBu", cbar_kws={"label": "Pre-monsoon count"}, ax=ax)

ax.set_title("Pre-Monsoon Counts: Species × Transect")
ax.set_xlabel("Transect")
ax.set_ylabel("Plant species")

plt.tight_layout()
plt.show()


### 6.5 Regression plot — is there a linear relationship?

In [ ]:
# sns.regplot draws a scatter plot PLUS a fitted linear regression line with
# a shaded confidence band — useful to see if pre- and post-monsoon counts move together
fig, ax = plt.subplots(figsize=(8, 5))

sns.regplot(data=df, x="pre monsoon", y="post monsoon", ax=ax,
            scatter_kws={"alpha": 0.5}, line_kws={"color": "red"})

ax.set_title("Regression: Post-Monsoon vs Pre-Monsoon Counts")
ax.set_xlabel("Pre-monsoon count")
ax.set_ylabel("Post-monsoon count")

plt.tight_layout()
plt.show()


In [ ]:
# We can also get the exact correlation coefficient (a number between -1 and 1)
# .corr() measures how strongly two numeric columns move together linearly
correlation = df["pre monsoon"].corr(df["post monsoon"])
print(f"Correlation between pre- and post-monsoon counts: {correlation:.2f}")


### 6.6 Pairplot — one chart, every pairwise relationship

`pairplot` is a quick way to see relationships between ALL numeric columns at once: histograms on the diagonal, scatter plots everywhere else.

In [ ]:
# hue="Transect" colors each point by transect, so we can also compare groups visually
# NOTE: pairplot can be slow on large datasets since it creates a full grid of plots
sns.pairplot(df, vars=["pre monsoon", "post monsoon", "change"], hue="Transect", palette="tab10")

plt.suptitle("Pairwise Relationships between Count Variables", y=1.02)
plt.show()


## 7. Putting it together — a mini end-to-end analysis

Let's answer one concrete question: **"Which species showed the biggest increase and decrease after the monsoon, and how does that vary by transect?"**

In [ ]:
# Step 1: find the single biggest increase and decrease in the whole dataset
biggest_increase = df.loc[df["change"].idxmax()]
biggest_decrease = df.loc[df["change"].idxmin()]

print("Biggest increase:")
print(biggest_increase[["Transect", "PLANT", "pre monsoon", "post monsoon", "change"]])

print("\nBiggest decrease:")
print(biggest_decrease[["Transect", "PLANT", "pre monsoon", "post monsoon", "change"]])


In [ ]:
# Step 2: visualize the top 5 increases and top 5 decreases together
top5_increase = df.nlargest(5, "change")      # nlargest: rows with the 5 highest values
top5_decrease = df.nsmallest(5, "change")     # nsmallest: rows with the 5 lowest values

# Combine them into a single small DataFrame for plotting
combined = pd.concat([top5_increase, top5_decrease])

fig, ax = plt.subplots(figsize=(10, 6))

# Color bars differently depending on whether the change was positive or negative
colors = ["seagreen" if c > 0 else "indianred" for c in combined["change"]]

ax.barh(combined["PLANT"], combined["change"], color=colors)

ax.axvline(0, color="black", linewidth=0.8)   # vertical line at 0 for reference
ax.set_title("Biggest Increases and Decreases in Plant Counts After Monsoon")
ax.set_xlabel("Change (post − pre)")
ax.set_ylabel("Plant species")

plt.tight_layout()
plt.show()


## 8. Summary — what we learned

**Pandas**
- Loading data with `read_excel`, inspecting with `.head()`, `.info()`, `.describe()`
- Cleaning column names and checking for missing/duplicate values
- Filtering rows with boolean masks, creating derived columns
- `groupby()` for split-apply-combine aggregation
- `pivot_table()` for reshaping data into a grid

**Matplotlib**
- The Figure/Axes model (`plt.subplots()`)
- Bar charts, histograms, scatter plots, and multi-panel subplots
- Always labeling titles, axes, and legends

**Seaborn**
- Boxplots & violin plots for comparing distributions across categories
- Bar plots with automatic confidence intervals
- Heatmaps for visualizing 2D tables (like our pivot table)
- Regression plots (`regplot`) and pairwise plots (`pairplot`) for relationships between variables

### Next steps to keep practicing
- Try grouping by `PLANT` instead of `Transect` and see which species are most volatile
- Explore `sns.catplot` and `sns.FacetGrid` for more complex multi-panel statistical plots
- Try exporting a cleaned version of `df` back to Excel with `df.to_excel("cleaned_data.xlsx", index=False)`
